# **5G Deployment Example**

In this example, we will deploy a 5G network using the default 5G parameters of the Post-5G orchestrator, which are:

- Open5GS core with: AMF, AUSF, BSF, NSSF, NRF, PCF, SMF, UPF, UDM, UDR
- OAI gNB
- A simulated OAI NR UE

First, we install the required dependencies:

In [ ]:
%pip install -e ../../.. --extra-index-url=https://gitlab.inria.fr/api/v4/projects/70914/packages/pypi/simple

In [ ]:
#! pip install --upgrade "git+https://gitlab.inria.fr/yamami/orchestrator.git@container-log-selection"

## Install Dependencies:

Install the SLICES CLI used for the BI to create VMs, and for POS to provision baremetal serevers:

In [ ]:
#! pip install slices-cli --extra-index-url=https://doc.slices-ri.eu/pypi/

Install the SLICES POS CLI:

In [ ]:
#! pip install slices-cli-pos --index-url https://gitlab.inria.fr/api/v4/projects/68732/packages/pypi/simple --extra-index-url https://doc.slices-ri.eu/pypi/

Authenticate with your SLICES account:

In [ ]:
#! slices auth login

Install the Post-5G orchestrator:

In [ ]:
%pip install -e ../../.. --extra-index-url=https://gitlab.inria.fr/api/v4/projects/70914/packages/pypi/simple

Install the Ansible dependencies:

In [ ]:
! ansible-galaxy install -r ../../../../ansible/requirements.yaml

Make sure you have an SSH key that you add to SLICES using the command:

```bash
slices pubkey register /path/to/ssh/key
```

which will be use to grant SSH access to the provisioned SLICES machines.

## Input 5G Network Description:

The Post-5G orchestrator keeps the logical network description separate from the deployment-specific choices and evolving deployment state, so the same experiment can be adapted to different implementations or <YOUR_PASSWORD>beds.

To achieve that, it uses two separate models:
- The Post-5G metamodel describing the entire network and its relationships using in pure 3GPP-standard way
- The deployment model describing implementation choices, target nodes, provider settings, and operational state

To deploy, you invoke lifecycle managers for the work you need. For example, the `ProvisioningManagement` class reserves or creates resources, `KubernetesManagement` prepares nodes and forms a cluster, `CoreManagement`, `RANManagement`, `RICManagement`, and `UEManagement` prepare and install workloads on their respective 5G components.

These stages are composable management operations, not necessarily one automatic "deploy everything" command.

Each stage exposes its particular set of actions that run the corresponding Ansible roles and Helm actions, often concurrently where possible.

To get a Meta Model, we can initiate an empty `MetaModel` object that will use a default set of parameters to provide a 3GPP-standard 5G network description:

In [ ]:
from metamodel.models.metamodel import MetaModel

metamodel = MetaModel()

To see the full content of this default Meta Model:

In [ ]:
print(metamodel.to_yaml(exclude_defaults=False))

By default, the metamodel contains:

- Open5GS core with: AMF, AUSF, BSF, NSSF, NRF, PCF, SMF, UPF, UDM, UDR
- OAI gNB
- A simulated OAI NR UE

Next, to get the deployment model, we need initiate a `DeploymentModel` object.

The easiest way to do so is by providing a YAML input that describes our deployment.

By default, the `DeployemntModel` defines 4 large-size VMs on the SLICES BI `fr-sophia2-bi-vm1` infrastructure.

However, we will have to modify some parameters to ensure the VM provisioning is configured correctly.

In [ ]:
from orchestrator.models.deployment import DeploymentModel
from metamodel.models.metamodel import MetaModel

metamodel = MetaModel()
deployment_model = DeploymentModel.from_yaml(
"""
params:
  provisioning:
    project: post5g-beta
    experiment: 5g
    extra_path: ~/slices/bin
"""
)

Here, we specified:
- The absolute path to where the SLICES CLI was installed on the Python venv
- The SLICES project to use: `post5g-beta`
- The SLICES experiment: `5g`

Make sure to create the SLICES experiment beforehand using:

```bash
slices project use post5g-beta
slices experiment create 5gbp-exp --duration 1d
```

or add `create_experiment: true` to your deployment model `params.provisioning` to create the experiment automatically.

By default, the created experiment duration is 1 day, and the VM duration is 3 hours. To change that, use the `experiment_duration` and `node_duration` parameters respectively.

> *Note*: Check if your Ansible roles path was found automatically:

In [ ]:
deployment_model.params.ansible.roles_paths

> *Note*: If the list is empty, your Ansible roles path wan not found. If so, before continuing, please also add the absolute path to your Ansible roles in the `params.ansible.roles_paths` list.

## Provisioning:

To provision the deployment VMs, we will invoke the `ProvisioningManagement` class from the Post-5G orchestrator and initiate it with the `metamodel` and `deployment_model`:

In [ ]:
from orchestrator.deployment import ProvisioningManagement

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)

> *Note*: During the initialization phase of each deployment stage management class, a reconcilation phase ensures the deployment model is  synchronized with the meta model, which takes precedence. 

To provision the nodes, we run the `provision()` action provided by this deployment management class, which will internally run the corresponding `provisioning/bi/provision` Ansible role:

In [ ]:
await prov.provision()

> *Note*: All the deployment actions are asynchronous function that have to be awaited.

In [ ]:
prov.output_merged(path="merged.yaml", output="yaml")

Once the provisioning is finished, the deployment model will be updated to contain the SSH informations about the provisoned nodes needed to generate an Ansible inventory used for the subsequent deployement stages.

In [ ]:
print(prov.get_ansible_inventory())

You can run commands on the provisioned nodes using the the `run_node_command` function. For example:

In [ ]:
print((await prov.run_node_command(command=["apt", "update"], nodes=["node1"], become=True, quiet=True))["node1"])

> *Note*: The `run_ue_command` can abe used to run commands on provisioned COTS UEs that allow SSH access.

## Kubernetes Deployment:

Next, to deploy Kuberenetes on the provisioned nodes, we will invoke the `KubernetesManagement` class.

To initiate it, we can either use the `output()` function of the previous deployment stage to get the metamodel (which is unchanged) and the deployment model (which now contains the SSH info about the nodes) output, and then use them as input just like we did before for the `ProvisioningManagement`.

Or more conveniantly, we can give the entire previous deployment class as input for the next one like this:

In [ ]:
from orchestrator.deployment import K8SManagement

k8s = K8SManagement(preceding=prov)

To deploy Kuberenetes, we will call the `deploy()` action fucntion of the `KubernetesManagement` class, which is a wrapper fucntion that internally calls the `prepare()` actions to install the necessary dependencies on all the provisioned nodes, then the `create()` action to initiate the Kubernetes cluster on the master node (by default the first node in the `nodes` section of the deployment model) and get the Kuberenetes join command, and finally the `join()` action to add the remaining worker nodes on the newly created cluster.

Again, these actions execute internally the corresponding `k8s/<action>` ansible role.

In [ ]:
await k8s.deploy()

Once the Kubernetes cluster is created we can move to the actual 5G deployment.

## 5G Core Deployment:

To deploy the 5G core network, we again invoke the corresponding `CoreManagement` class and initiate it using its preceding class: 

In [ ]:
from orchestrator.deployment import CoreManagement

core = CoreManagement(preceding=k8s)

> *Note*: The 5G deployment management classes (`CoreManagement`, `RANMamanegemt`, `RICManagement` and `UEManagement`) all have a reconcilation phase that handles resolving the cross-references between the different 5G components (e.g. gNB referencing the AMF) and populating the container images and other Kuberenetes parameters that can also be changed manually through the deployment model.

Again, we call the `deploy()` action fucntion, which internally will prepare the core network components (AMF, AUSF, BSF, NSSF, NRF, PCF, SMF, UPF, UDM, UDR) and the database by:
- copying the Helm chart file to the master node
- rendering the Jinja templates from the metamodel and deployment model configuration parameters
- installing the Helm chart and getting the installed pod network information
- registering the subscribers in the `ues` section of the metamodel on the 5G core database

### Select the metrics-enabled SMF and UPF images

Run this cell before the following core deployment cell. It prints the selected SMF and UPF images.


In [ ]:
# Use the metrics image for SMF slice mappings and UPF traffic counters.
from metamodel.models.core import CoreComponentType
from orchestrator.models.k8s import ContainerImage
from orchestrator.models.core import ADVANCED_CORE_CONFIG_MAPPING, CoreImplementation, DeployedCoreComponent
from orchestrator.utils.defaults import DEFAULT_CORE_IMAGES

for component in core.deployment_model.core.components.values():
    if component.implementation != CoreImplementation.OPEN5GS:
        continue
    advanced_type = ADVANCED_CORE_CONFIG_MAPPING[component.implementation][component.type]
    advanced = component.advanced_config.model_dump() if component.advanced_config is not None else {}
    if component.type in (CoreComponentType.SMF, CoreComponentType.UPF):
        component.image = ContainerImage(
            repository="ghcr.io/niloysh/open5gs",
            image_tag="v2.7.0-upf-metrics-v2",
        )
        component.open5gs_binary_dir = "/open5gs/install/bin"
        advanced["logger"] = {"file": f"/tmp/open5gs-{component.type}.log", "level": "info"}
        if component.type == CoreComponentType.UPF:
            advanced.setdefault("teid_range_indication", 0)
    else:
        component.image = DEFAULT_CORE_IMAGES[component.implementation][component.type].model_copy(deep=True)
        component.open5gs_binary_dir = DeployedCoreComponent.model_fields["open5gs_binary_dir"].default
        if advanced.get("logger") == {"file": f"/tmp/open5gs-{component.type}.log", "level": "info"}:
            advanced.pop("logger")
    component.advanced_config = advanced_type.model_validate(advanced)

for name in ("smf", "upf"):
    image = core.deployment_model.core.components[name].image
    print(f"{name.upper()} image selected: {image.repository}:{image.image_tag}")


In [ ]:
await core.deploy()

## RAN Deployment:

Once the 5G core has beed deployed, the deployment model should now contain the necessary network information about the AMF for the gNB to connect to it.

To deploy the OAI RFSIM RAN, we again call the corresponding `RANManagement` class and initiate it using the preceding class:

In [ ]:
from orchestrator.deployment import RANManagement

ran = RANManagement(preceding=core)

Then call the `deploy()` action function to prepare and install the gNB Helm chart:

In [ ]:
# Enable the statistics-file exporter before the gNB pod is created.
from orchestrator.models.ran import OAIAdvancedGNBConfig

gnb = ran.deployment_model.ran.components["gnb"]
gnb.advanced_config = OAIAdvancedGNBConfig.model_validate(
    gnb.advanced_config.model_dump() if gnb.advanced_config is not None else {}
)
gnb.advanced_config.metrics_parser.enabled = True
await ran.deploy()

For all the available 5G deployement management classes, we can use the `get_logs()` action function that will fetch logs from the deployed 5G components. A single component returns text; multiple components return a dictionary keyed by component name.

> *Note*: By default, the orchestrator fetches the logs from the deployed 5G component since the last 1 hour, but again, these paramaters can be changed for each component in the deployement model.

In [ ]:
print(await ran.list_containers())
print(await ran.get_logs(
    components=["gnb"],
    container="gnb",
    quiet=True,
))

> *Note*: The `quiet=True` option is used to supress the Ansible output to stdout.

## Monitoring before UE attachment

The core and RAN must be running before monitoring is installed. AMF/SMF capture
must be active **before the first UE attaches**, so the mapper sees registration
and session setup. Do not run the UE deployment cells before this section passes.

Create `monitoring.yaml` beside this notebook using the configuration in
[the integration guide](../../../../MONITORING.md). Enable `latency: true` and
`packet_pairing: true`; verify actual AMF/SMF/UPF/gNB container names and capture
interfaces. The Grafana Helm chart manages the default `admin` / `admin` credentials.
`prepare()` resolves namespaces and selectors from the deployed model.

The experiments open their own Prometheus SSH tunnels using the deployment model.
Set the artifact query profile in `QUERIES_FILE` for the iperf experiment.


In [ ]:
from pathlib import Path
import asyncio
import json

from orchestrator.deployment import MonitoringManagement
from orchestrator.models.monitoring import MonitoringDeployment

QUERIES_FILE = Path("queries.json")  # Your Prometheus export query profile.

monitoring = MonitoringManagement(preceding=ran)
monitoring.deployment_model.monitoring = MonitoringDeployment.from_yaml_file("monitoring.yaml")

In [ ]:
try:
    await monitoring.deploy()
finally:
    monitoring.output(
        metamodel_file="monitoring_metamodel.yaml",
        deployment_model_file="monitoring_deployment_model.yaml",
    )

### Open Grafana from this notebook

Run after monitoring deployment. This forwards localhost port **30096** to the
current control VM's Grafana port **32005**, using the model's SSH jump host.
The tunnel stays open while you use Grafana. Rerunning the cell replaces its
previous tunnel. To close it explicitly, run `grafana_tunnel.terminate()`.


In [ ]:
import atexit
import shlex
import subprocess
from IPython.display import Markdown, display

# Close the previous tunnel before opening one with the current host address.
if "grafana_tunnel" in globals() and grafana_tunnel.poll() is None:
    grafana_tunnel.terminate()
    await asyncio.to_thread(grafana_tunnel.wait, 5)

# get SSH info and fallback
cfg = monitoring.deployment_model.monitoring
control = cfg.installed_control_node or cfg.control_node or monitoring.deployment_model.params.k8s.master_node
ssh = monitoring.deployment_model.nodes[control].ssh_info

command = ["ssh", "-N", "-T", "-S", "none", "-o", "ControlMaster=no", "-o", "ControlPersist=no",
           "-o", "BatchMode=yes", "-o", "ExitOnForwardFailure=yes", "-o", "ConnectTimeout=10"]

command += shlex.split(ssh.common_args or "")

if ssh.private_key_file:
    command += ["-i", str(Path(ssh.private_key_file).expanduser())]
command += ["-p", str(ssh.port), "-L", f"127.0.0.1:30096:127.0.0.1:{cfg.grafana_node_port}",
            f"{ssh.user}@{ssh.hostname or control}"]

grafana_tunnel = subprocess.Popen(command, stdout=subprocess.DEVNULL)

# Close the tunnel when the kernel exits normally.
atexit.register(grafana_tunnel.terminate)

GRAFANA_URL = "http://127.0.0.1:30096"
display(Markdown(f"[Open Grafana]({GRAFANA_URL}) — username: **admin**, password: **admin**"))


## UE Deployement:

To deploy the OAI RFSIM NR UE defined in the meta and deployemnt models, we again initiate the corresponding `UEManagement` deployement management class from the previous one to maintain the deployment state so far, which includes the necessary gNB pod network information that will be used to configure and deploy the simulated UE correctly:

In [ ]:
from orchestrator.deployment.ue import UEManagement

ue = UEManagement(preceding=monitoring)


To deploy, we again call the `deploy()` action function:

In [ ]:
await ue.deploy()

Once deployed, we can again use the `get_logs()` function to check the logs of our UE deployment whitout having to manually access the Kuberenetes pod:

In [ ]:
print(await ue.get_logs(components=["sim_ue"], quiet=True))

### Check the first UE attachment

Inspect the UE logs above, then require an IPv4 address on the simulated UE's
tunnel interface. Adjust `UE_INTERFACE` if your UE image uses another name.
A tunnel address alone does not prove connectivity or correct monitoring data.


In [ ]:
UE_INTERFACE = "oaitun_ue1"
attachment = await ue.run_command(
    command=["ip", "-4", "addr", "show", "dev", UE_INTERFACE],
    components=["sim_ue"], quiet=True,
)
if "inet " not in attachment["sim_ue"]:
    raise RuntimeError("The first UE has no IPv4 tunnel address; inspect attachment logs")
print(attachment["sim_ue"])


To SSH into the SLICES BI VM and check your deployement, you can run:

```bash
slices bi --infra fr-sophia2-bi-vm1 ssh node1
```

> Make sure you use the same SLICES experiment name used in the deployment model.

From there, you can do:

```bash
sudo su -
```

to switch to the priviledged user, and use the standard `kubectl` commands to check your deployment.

In this case, the command to access the UE pod will be:

```bash
kubectl exec deployment/oai-sim-ue -n oai-ue -it -- bash
```

> *Note*: Each 5G component defined in the meta and deployement model follows these naming conventions:
> - namespace: `<implementation>-<category>` (e.g. AMF is in the `core` category and is deployed using the `Open5GS` implemetation, so it will be in the `open5gs-core` namespace)
> - chart name: `<implementation>-<name>` (e.g. we deployed an `OAI` gNB called `gnb`, so its deployement name will be `oai-gnb` in the namespace `oai-ran`)

A more convenient option is to use the `run_command` from the deployment management class to execute a command directly on the pod. For example:

In [ ]:
print((await ue.run_command(command=["ip", "addr"], quiet=True))["sim_ue"])

## Adding a new component on the fly:

To add a new 5G component to the metamodel, we can use the `add_component()` action funntion in the 5G deployemnt management classes to add a new corresponfing 5G component. For example, we can add a UE from the `UEManagement` class we have.

For that, we can use the `UE` meta model to specify a `supi` field different from the default value of `001010000000100`:

In [ ]:
from metamodel.models.ue import UE

ue.add_component(name="sim_ue2", component_meta=UE.from_yaml(
"""
config:
    supi: '001010000000101'
"""
))

This action only adds the UE to the metamodel and calls the synchronize internally to update the deployment model.

To actually add the UE to the 5G network, we first have to register it in the database using the `register_ues()` action function in the `CoreManagement` class:

In [ ]:
core = CoreManagement(preceding=ue)

In [ ]:
await core.register_ues(ues=["sim_ue2"], quiet=True)

Next, we deploy the new `sim_ue2` using the `deploy` action in the `UEManagement` class again:

In [ ]:
# Carry the subscriber-registration stage forward as well.
ue = UEManagement(preceding=core)
await ue.deploy(components=["sim_ue2"], quiet=True)


In [ ]:
# Run only after the traffic experiments: await ue.remove(components=["sim_ue2"])

Once finished, we fetch and print the UE's logs to ensure the deployment was successfull:

In [ ]:
print(await ue.get_logs(components=["sim_ue2"], quiet=True))

## TCP iperf from UEs to a Server

Choose `SERVER_NODE` from the deployment model; the default is `node1`.
The setup cell installs iperf3 on the server VM when needed. Its current SSH address is used as the traffic
endpoint, while each UE binds to its own tunnel address and interface.
Each UE runs 20 seconds uplink, then 20 seconds downlink using reverse mode,
sequentially. One-off VM servers use separate ports starting at 5202, leaving
port 5201 available for an existing iperf service.

The setup cell reads the addresses and ensures the server package is installed. The experiment
cell shows all traffic commands and the artifact collector explicitly. Server
startup is outside traffic windows; JSON reports, configuration, timeline and
Prometheus exports share one result directory. Collection finishes before the
tunnel closes, including on traffic failure. No latency-counter assertion is required.


In [ ]:
from datetime import UTC, datetime
from orchestrator.artifacts import ExperimentRun
from orchestrator.utils.ansible import get_ansible_inventory_line, run_ansible_command, run_ansible_playbook

SERVER_NODE = "node1"
CLIENTS = ["sim_ue", "sim_ue2"]
DURATION, BASE_PORT = 20, 5202
BASELINE_SECONDS, COLLECTION_SECONDS = 5, 5
UE_INTERFACE = "oaitun_ue1"

server_node = ue.deployment_model.nodes[SERVER_NODE]
server_ip = server_node.ssh_info.hostname
server_inventory = get_ansible_inventory_line(SERVER_NODE, server_node)
server_runner = str(Path(ue.deployment_model.params.ansible.runner_path) / "iperf-server")


await run_ansible_playbook(
    playbook=[{
        "name": "Install iperf3 on the server VM", "hosts": SERVER_NODE,
        "gather_facts": False, "become": True,
        "tasks": [{"name": "Install iperf3", "ansible.builtin.apt": {
            "name": "iperf3", "state": "present", "update_cache": True,
            "cache_valid_time": 3600,
        }}],
    }],
    inventory=server_inventory, runner_path=server_runner, quiet=True,
)
addresses = await ue.run_command(
    command=["ip", "-j", "-4", "addr", "show", "dev", UE_INTERFACE],
    components=CLIENTS, quiet=True,
)

ue_ips = {}
for name in CLIENTS:
    ipv4 = [a["local"] for device in json.loads(addresses[name]) for a in device["addr_info"]
            if a["family"] == "inet"]
    if len(ipv4) != 1:
        raise RuntimeError(f"{name}: expected one IPv4 address on {UE_INTERFACE}")
    ue_ips[name] = ipv4[0]

if len(set(ue_ips.values())) != len(CLIENTS):
    raise ValueError("Select UEs with different tunnel addresses")

if not QUERIES_FILE.is_file():
    raise FileNotFoundError(QUERIES_FILE)

print(f"iperf server: {SERVER_NODE} ({server_ip}); UE clients: {ue_ips}")


### 1. Define the traffic commands




In [ ]:
async def start_iperf_server(port):
    await run_ansible_command(
        command=["iperf3", "-s", "-D", "-1", "--idle-timeout", "60",
                 "-B", server_ip, "-p", str(port), "-J"],
        hosts=SERVER_NODE,
        inventory=server_inventory,
        runner_path=server_runner,
        quiet=True,
    )


async def iperf_client(client, direction, duration, port):
    reverse = {"uplink": [], "downlink": ["-R"]}[direction]
    output = await ue.run_command(
        command=["iperf3", "-c", server_ip, "-B", ue_ips[client],
                 "--bind-dev", UE_INTERFACE, "-p", str(port), "-t", str(duration),
                 "--connect-timeout", "10000", "-J", "--get-server-output", *reverse],
        components=[client],
        quiet=True,
    )
    return json.loads(output[client])


### 2. Define your scenario

Each row is one measured step. Add, remove or reorder rows, select a UE and direction,
or set a different duration in seconds. Steps execute sequentially.


In [ ]:
SCENARIO = [
    {"name": "ue1_uplink",   "client": "sim_ue",  "direction": "uplink",   "duration": DURATION},
    {"name": "ue1_downlink", "client": "sim_ue",  "direction": "downlink", "duration": DURATION},
    {"name": "ue2_uplink",   "client": "sim_ue2", "direction": "uplink",   "duration": DURATION},
    {"name": "ue2_downlink", "client": "sim_ue2", "direction": "downlink", "duration": DURATION},
]


### 3. Collect artifacts while executing the scenario

`ExperimentRun` owns the result directory, timeline and configured Prometheus export.
`section(name)` marks a measurement window; `save_json` saves a report in that run.
The server starts before each traffic window. The final wait allows additional
scrapes. The collector opens its own Prometheus tunnel during artifact export
and closes it afterward.

For another workload, use the same `with run.section("step_name"):` pattern around
its command and save its result. The collector itself does not depend on iperf.


In [ ]:
timestamp = datetime.now(UTC).strftime("%Y-%m-%d_%H-%M-%S.%fZ")
output_directory = Path("results") / f"iperf-{timestamp}"

async with ExperimentRun(
    output_directory,
    name="ue-to-vm-iperf",
    monitoring=monitoring,
    queries_file=QUERIES_FILE,
) as experiment:
    experiment.save_json("scenario.json", SCENARIO)

    with experiment.section("baseline"):
        await asyncio.sleep(BASELINE_SECONDS)

    for port, <YOUR_PASSWORD> in enumerate(SCENARIO, start=BASE_PORT):
        print(f"Running {<YOUR_PASSWORD>['name']}: {<YOUR_PASSWORD>['client']}, {<YOUR_PASSWORD>['direction']}, {<YOUR_PASSWORD>['duration']} seconds")
        await start_iperf_server(port)

        with experiment.section(<YOUR_PASSWORD>["name"]):
            report = await iperf_client(
                client=<YOUR_PASSWORD>["client"],
                direction=<YOUR_PASSWORD>["direction"],
                duration=<YOUR_PASSWORD>["duration"],
                port=port,
            )

        experiment.save_json(f"{<YOUR_PASSWORD>['name']}.json", report)

    with experiment.section("collection"):
        await asyncio.sleep(COLLECTION_SECONDS)

print(f"Artifacts saved to: {output_directory.resolve()}")


To retreive the metamodel and deployement model output of your deployement managememt classes, you can use the `output()` function:

In [ ]:
ue.output(metamodel_file="output_metamodel.yaml", deployment_model_file="output_deployment_model.yaml")

> *Note*: All your Ansible run logs are stored by action, component and execution timestamp in `~/post-5g/ansible-runner` by default, which you can of course change using the `runner_path` parameter in the `params.ansible` section of the deployment model.

To destroy the VMs, we can again invoke the `ProvisioningManagement` class and call the `unprovision` action fucntion:

In [ ]:
# await prov.unprovision()

> *Note*: most actions have an opposite action to undo what the original action has done, for example, we have the "remove" action to undo the "deploy" action, the "unprovision" action to undo the "provision" action ... etc.

## Miscellaneous:

The `Miscellaneous` class is the Post-5G orchestrator is a flexible extension point.

It runs custom Ansible playbooks against the current deployment, supplying the merged metamodel and deployment model by default, as well as the default Ansible inventory files used by the built-in roles.

It can collect selected playbook outputs and store them in the deployment model's `miscellaneous` section, to allow experiment-specific tasks that don’t belong to the standard provisioning, Kubernetes, core, RAN, RIC, or UE stages.